In [5]:
import torch
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("/sujin/PycharmProjects/Pretraining")

import numpy as np
import random
import json

from tqdm import tqdm
from Bio import SeqIO
from utils.fetch_uniprot import fetch_sequence
from utils.others import setup_seed, progress_bar
from utils.generate_lmdb import dump_lmdb, get_value
from dataset.ted.ted_dataset import TedDataset
from model.ted.ted_domain_model import TedDomainModel
from utils.embedding_collector import EmbeddingCollector
from utils.protein_util import calc_seq_identity
from utils.others import setup_seed

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
model_tag = "TED_650M"

config = {
    "config_path": f"/sujin/Models/TED/{model_tag}",
    "from_checkpoint": f"/sujin/Models/TED/{model_tag}/{model_tag}.pt",
}

device = "cuda"
model = TedDomainModel(**config)
model.to(device)

No lr_scheduler_kwargs provided. The default learning rate is 0.
No optimizer_kwargs provided. The default optimizer is AdamW.
Some weights of the model checkpoint were not used: ['esm.embeddings.position_ids']


TedDomainModel(
  (model): EsmForMaskedLM(
    (esm): EsmModel(
      (embeddings): EsmEmbeddings(
        (word_embeddings): Embedding(446, 1280, padding_idx=1)
        (dropout): Dropout(p=0.0, inplace=False)
        (position_embeddings): None
      )
      (encoder): EsmEncoder(
        (layer): ModuleList(
          (0-32): 33 x EsmLayer(
            (attention): EsmAttention(
              (self): EsmSelfAttention(
                (query): Linear(in_features=1280, out_features=1280, bias=True)
                (key): Linear(in_features=1280, out_features=1280, bias=True)
                (value): Linear(in_features=1280, out_features=1280, bias=True)
                (dropout): Dropout(p=0.0, inplace=False)
                (rotary_embeddings): RotaryEmbedding()
              )
              (output): EsmSelfOutput(
                (dense): Linear(in_features=1280, out_features=1280, bias=True)
                (dropout): Dropout(p=0.0, inplace=False)
              )
              (La

# Generate LMDB files

In [6]:
seg_path = "/sujin/Datasets/TED/sa_segments_ge2.tsv"
uniprot2idx = {}
idx2uniprot = {}
idx2seg = {}
seg2idx = {}
with open(seg_path, "r") as r:
    for line in tqdm(r):
        uniprot_id, sa_seg = line.strip().split("\t")
        idx = seg2idx.get(sa_seg, len(seg2idx))
        seg2idx[sa_seg] = idx
        idx2seg[idx] = sa_seg
        uniprot2idx[uniprot_id] = uniprot2idx.get(uniprot_id, []) + [idx]
        idx2uniprot[idx] = idx2uniprot.get(idx, []) + [uniprot_id]

uniprot2idx_lmdb = "/sujin/Datasets/TED/sa_segments_ge2_lmdb/uniprot2idx"
dump_lmdb(uniprot2idx, uniprot2idx_lmdb)

idx2uniprot_lmdb = "/sujin/Datasets/TED/sa_segments_ge2_lmdb/idx2uniprot"
dump_lmdb(idx2uniprot, idx2uniprot_lmdb)

idx2seg_lmdb =  "/sujin/Datasets/TED/sa_segments_ge2_lmdb/idx2seg"
dump_lmdb(idx2seg, idx2seg_lmdb)

265616762it [18:40, 237098.70it/s]
Dumping data...: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 260637958/260637958 [25:57<00:00, 167347.15it/s]


In [9]:
# Only record seg2idx, since this dict cannot be saved through lmdb.
seg_path = "/sujin/Datasets/TED/sa_segments_ge2.tsv"
seg2idx = {}
with open(seg_path, "r") as r:
    for line in tqdm(r):
        uniprot_id, sa_seg = line.strip().split("\t")
        if sa_seg not in seg2idx:
            idx = len(seg2idx)
            seg2idx[sa_seg] = idx

265616762it [08:26, 524777.02it/s]


In [9]:
for idx, sa_seg in idx2seg.items():
    print(idx, sa_seg)
    print(seg2idx[sa_seg])
    break

0 TcGvDvRvKdRdEdFdLfEdLwGdRcKdAqGvRdFpPpAwAiSdTgSpNvGgEiIfSgIeWf<unk>ElEsRnRvRqPlAlEvNqAlRvLlTlHvGvLlLcRvEvRlDlIfPdVfLsSdDsRnShPsIwVtPwVgLwVqGqEaDdRvMlClKvRqMlSqAvLqPcLcEvRpHvGsAyYhVwQdAwIdDdApPpSnVdPdArGrErEiItLgRtIgArPrShAsVsHdEdTpEvEnIsHvRvFsVsDvAsLsDsGvIsWcSvEvLsGv
0


In [ ]:
setup_seed(20000812)

# Randomly sample 100000 segments
inds = random.sample(list(idx2seg.keys()), 300000)

sa_segs = [idx2seg[i] for i in inds]
save_path = "/sujin/Datasets/TED/embedding/2026-01-27-TED-650M-plddt70/random_300k_sa_segs.tsv"
with open(save_path, "w") as w:
    for sa_seg in sa_segs:
        w.write(f"{sa_seg}\n")

# Get UniProt IDs of randomly sampled segments

In [ ]:
idx2uniprot_lmdb = "/sujin/Datasets/TED/sa_segments_ge2_lmdb/idx2uniprot"

for sa_seg in sa_segs:
    idx = seg2idx[sa_seg]
    uniprot_ids = eval(get_value(idx2uniprot_lmdb, sa_seg))

    # Choose the first UniProt ID
    print(uniprot_ids)

    break

# Start retrieval

In [4]:
def do(process_id, idx, model, sa_seg):
    with torch.no_grad():
        key_repr = model.get_key_repr(sa_seg)
    return key_repr.cpu().numpy()


sa_segs = []
sa_seg_path = "/sujin/Datasets/TED/embedding/2025-10-20-TED-650M-allData/random_300k_sa_segs.tsv"
with open(sa_seg_path, "r") as r:
    for line in r:
        sa_segs.append(line.strip())

save_path = "/sujin/Datasets/TED/embedding/2025-10-20-TED-650M-allData/random_300k_sa_segs.npy"
embedding_collector = EmbeddingCollector(data=sa_segs, model=model, do=do, save_path=save_path, return_type="np", n_process=1)
embedding_collector.run()

[Errno 25] Inappropriate ioctl for device
Can't get terminal size, set terminal_y = None


/root/miniconda3/envs/pl/lib/python3.9/site-packages/torch/utils/checkpoint.py:61: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


7Total: [#######################################] 100% 300000/300000 [03:34:55 < 00:00:00, 23.26it/s]8

Aggregating embeddings...: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.39it/s]


array([[-0.03627491, -0.04937655,  0.02451828, ...,  0.0112561 ,
         0.0194981 ,  0.01626207],
       [-0.05165455, -0.07160645,  0.05924021, ...,  0.00655675,
         0.02865911, -0.01782134],
       [-0.01847178, -0.06670295,  0.01778607, ...,  0.02418606,
         0.01804018, -0.03341734],
       ...,
       [-0.02768051, -0.04419088,  0.0057795 , ...,  0.03878551,
         0.00841342, -0.00049019],
       [-0.02414593, -0.04116145,  0.04900422, ...,  0.03783351,
         0.03084244,  0.00956237],
       [-0.02810227, -0.02511443,  0.02733326, ...,  0.03568095,
         0.02417225,  0.00600075]], dtype=float32)

In [6]:
embedding_dir = "2025-10-20-TED-650M-allData"

sa_seg_path = f"/sujin/Datasets/TED/embedding/{embedding_dir}/random_300k_sa_segs.tsv"
sa_segs = []
with open(sa_seg_path, "r") as r:
    for line in r:
        sa_segs.append(line.strip())
print(len(sa_segs))


sa_seg_embedding_path = f"/sujin/Datasets/TED/embedding/{embedding_dir}/random_300k_sa_segs.npy"
sa_seg_embeddings = np.load(sa_seg_embedding_path)
sa_seg_embeddings = torch.tensor(sa_seg_embeddings).to(device)
print(sa_seg_embeddings.shape)

300000
torch.Size([300000, 1280])


In [10]:
uniprot2idx_lmdb = "/sujin/Datasets/TED/sa_segments_ge2_lmdb/uniprot2idx"
idx2uniprot_lmdb = "/sujin/Datasets/TED/sa_segments_ge2_lmdb/idx2uniprot"
idx2seg_lmdb =  "/sujin/Datasets/TED/sa_segments_ge2_lmdb/idx2seg"

save_path = "/sujin/Datasets/TED/embedding/2025-10-20-TED-650M-allData/retrieval_results-sameQuery.tsv"
total = 1000
with torch.no_grad(), open(save_path, "w") as w:
    w.write("Query_Segment\tCandidate_Segment\tThreshold_Score\tMatching_Score\tMax_Seq_Identity\tGround_Truth\tCandidate_Uniprot_IDs\n")
    
    cnt = 0
    for sa_seg in sa_segs:
        aa_seg = sa_seg.replace("<unk>", "")[::2]
        query_repr = model.get_query_repr(sa_seg).squeeze()
        matching_scores = torch.matmul(query_repr, sa_seg_embeddings.T) / model.temperature
        
        # Obtain the idx of the segment
        idx = seg2idx[sa_seg]
       
        # Get corresponding uniprot ids that contain this segment
        # uniprot_ids = idx2uniprot[idx]
        uniprot_ids = eval(get_value(idx2uniprot_lmdb, str(idx)))

        # Ensure this segment can compose a protein with two domains
        valid = False
        for uniprot_id in uniprot_ids:
            # if len(uniprot2idx[uniprot_id]) == 2:
            if len(eval(get_value(uniprot2idx_lmdb, str(uniprot_id)))) == 2:
                valid = True
                break

        if not valid:
            continue
        
        # Obtain all segments that can compose a protein with this segment
        label_inds = set()
        for uniprot_id in uniprot_ids:
            # for label_idx in uniprot2idx[uniprot_id]:
            for label_idx in eval(get_value(uniprot2idx_lmdb, str(uniprot_id))):
                if label_idx != idx:
                    label_inds.add(label_idx)
        
        # Calculate matching scores for these segments. Set a threshold based on these scores.
        threshold = 0
        for label_idx in label_inds:
            # label_seg = idx2seg[label_idx]
            label_seg = get_value(idx2seg_lmdb, str(label_idx))
            label_key_repr = model.get_key_repr(label_seg)
            label_score = torch.matmul(query_repr, label_key_repr.T) / model.temperature
            threshold = max(threshold, label_score.item())

        # Get candidate segments whose scores exceed the threshold
        # locations = (matching_scores >= threshold).nonzero().squeeze()
        # if len(locations.shape) == 0:
        #     locations = locations.unsqueeze(0)

        # Sort the matching scores and obtain the top-k candidates
        top_k = 10
        locations = torch.topk(matching_scores, k=top_k).indices

        for loc in locations:
            sa_seg_candidate = sa_segs[loc]
            aa_candi = sa_seg_candidate.replace("<unk>", "")[::2]
            
            # Compute sequence identity with label segments
            max_seq_iden = 0
            for label_idx in label_inds:
                # label_seg = idx2seg[label_idx]
                label_seg = get_value(idx2seg_lmdb, str(label_idx))
                aa_label = label_seg.replace("<unk>", "")[::2]
                seq_identity = calc_seq_identity(aa_candi, aa_label)
                max_seq_iden = max(max_seq_iden, seq_identity)
            
            # Only keep candidates with seq identity less than 0.5
            if max_seq_iden < 1:
                matching_score = matching_scores[loc]
                candi_idx = seg2idx[sa_seg_candidate]
                # candi_uniprot_ids = ",".join(idx2uniprot[candi_idx])
                candi_uniprot_ids = ",".join(eval(get_value(idx2uniprot_lmdb, str(candi_idx))))
                
                ground_truth = ",".join(uniprot_ids)
                w.write(f"{sa_seg}\t{sa_seg_candidate}\t{threshold:.4f}\t{matching_scores[loc].item():.4f}\t{max_seq_iden:.4f}\t{ground_truth}\t{candi_uniprot_ids}\n")
                # w.flush()
                
                cnt += 1
                progress_bar(cnt, total, desc="Saved retrieval results")
                
                if cnt == total:
                    break

            # For each query we only keep the first candidate
            break
            
        if cnt == total:
            break

Saved retrieval results [##################################################] 100% 1000/1000

In [ ]:
with torch.no_grad():
    for uniprot_id, inds in uniprot2idx.items():
        print(uniprot_id, inds)
        seq = sa_segs[inds[0]]
        true_hits = [sa_segs[i] for i in inds[1:]]
        
        query_repr = model.get_query_repr([seq])
        key_reprs = model.get_key_repr(true_hits)
        
        matching_score = torch.matmul(query_repr, key_reprs.T) / model.temperature
        print("True hits scores:", matching_score)
        
        for candidate in sa_segs[:100]:
            candidate_key_repr = model.get_key_repr([candidate])
            candidate_score = torch.matmul(query_repr, candidate_key_repr.T) / model.temperature
            print(f"Candidate: {candidate}, Score: {candidate_score.item()}")
        
        break